# ERA5-Land: quickstart

A minimal pull of ERA5-Land data: one hour of 2 metre temperature over the
UK at 0.1 degree (~9 km) resolution, opened with xarray and plotted.
Higher spatial detail than ERA5 single levels, land-only.

See [`docs/era5-land/README.md`](../docs/era5-land/README.md) for the full
reference. Prerequisites are the same as ERA5 single levels (CDS account,
licence accepted separately for ERA5-Land, `~/.cdsapirc` credentials).

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
VARIABLES = ["2m_temperature"]
YEARS = ["2023"]
MONTHS = ["07"]
DAYS = ["01"]
HOURS = ["12:00"]
BBOX = [55, -8, 49, 2]  # [north, west, south, east] - UK
OUTPUT_DIR = "../data/era5-land"
OUTPUT_FILENAME = "era5_land_quickstart.nc"
# ==================================================================

## Imports and environment check

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import cdsapi
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print(f"Python       {sys.version.split()[0]}")
for pkg in ["cdsapi", "xarray", "matplotlib", "cartopy", "netcdf4"]:
    print(f"{pkg:12} {version(pkg)}")

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError("Could not find repo root.")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from common.credentials import check_cds_credentials
check_cds_credentials()
print("\nCDS credentials found.")

## Submit the request

ERA5-Land requests are slightly simpler than ERA5 single levels: no
`product_type` field, and the dataset id is `reanalysis-era5-land`.

In [ ]:
output_path = Path(OUTPUT_DIR) / OUTPUT_FILENAME
output_path.parent.mkdir(parents=True, exist_ok=True)

request = {
    "variable": VARIABLES,
    "year": YEARS,
    "month": MONTHS,
    "day": DAYS,
    "time": HOURS,
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": BBOX,
}

client = cdsapi.Client()
client.retrieve("reanalysis-era5-land", request).download(str(output_path))
print(f"Saved: {output_path} ({output_path.stat().st_size / 1e6:.2f} MB)")

## Open and inspect

At 0.1 degree resolution over a UK bbox this produces roughly 60x100 grid
cells: substantially more detail than ERA5 single levels (25x41 over the
same area).

In [ ]:
ds = xr.open_dataset(output_path)
print(ds)

## Plot a map

ERA5-Land masks ocean cells; only land points have values. The finer grid
makes coastal detail and orographic gradients more visible than at the
0.25 degree ERA5 resolution.

In [ ]:
t2m_var = "t2m" if "t2m" in ds.data_vars else list(ds.data_vars)[0]
t2m = ds[t2m_var].squeeze() - 273.15

fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

t2m.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    cbar_kwargs={"label": "2 m temperature (C)"},
)

ax.coastlines(resolution="50m", linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="gray")
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False
gl.right_labels = False

ax.set_title(f"ERA5-Land 2 m temperature, {YEARS[0]}-{MONTHS[0]}-{DAYS[0]} {HOURS[0]} UTC")
plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/era5-land/README.md`](../docs/era5-land/README.md)
- **Compare with ERA5**: request the same variable from the single-levels endpoint and overlay the two. The higher resolution of ERA5-Land should be visible over coastlines and complex terrain.
- **Catchment-scale hydrology**: soil moisture, runoff, evaporation are natively at 9 km here, a useful step up from the 31 km ERA5 grid for small basins.
- **Aggregate to daily/monthly**: `ds.resample(valid_time="1D").mean()` as with ERA5.